# RuleShift Benchmark

Attach the packaged runtime, build the frozen leaderboard bundle, run the official task, and review the result.

In [ ]:
import sys
from pathlib import Path

import kaggle_benchmarks as kbench

RUNTIME_SRC = Path("/kaggle/input/datasets/raptorengineer/ruleshift-runtime/src")
if not RUNTIME_SRC.is_dir():
    raise FileNotFoundError(
        "Runtime dataset not found at /kaggle/input/datasets/raptorengineer/ruleshift-runtime/src -- "
        "attach the raptorengineer/ruleshift-runtime dataset."
    )
if str(RUNTIME_SRC) not in sys.path:
    sys.path.insert(0, str(RUNTIME_SRC))


In [ ]:
import pandas as pd

from tasks.ruleshift_benchmark.benchmark_bundle import (
    build_benchmark_bundle,
    build_leaderboard_rows,
)
from tasks.ruleshift_benchmark.runner import run_binary_task
from tasks.ruleshift_benchmark.splits import (
    MANIFEST_VERSION,
    discover_private_dataset_root,
)

PRIVATE_DATASET_ROOT = discover_private_dataset_root()
bundle = build_benchmark_bundle(
    include_private=PRIVATE_DATASET_ROOT is not None,
    private_dataset_root=PRIVATE_DATASET_ROOT,
)
leaderboard_df = pd.DataFrame(build_leaderboard_rows(bundle))
if leaderboard_df.empty:
    raise ValueError("leaderboard_df cannot be empty")

partition_df = pd.DataFrame(
    [
        {"split": partition["partition"], "episodes": partition["episode_count"]}
        for partition in bundle["partitions"]
    ]
)


In [ ]:
_status_df = pd.DataFrame(
    [
        ("Benchmark version", MANIFEST_VERSION),
        ("Episodes loaded", len(leaderboard_df)),
        ("Splits loaded", ", ".join(partition_df["split"])),
        ("Private dataset", "yes" if PRIVATE_DATASET_ROOT is not None else "no"),
    ],
    columns=["Field", "Value"],
)

_status_df


## Official task

`ruleshift_benchmark_v1_binary` is the leaderboard entry point. It loops over the frozen rows, scores each episode, and returns `(numerator, denominator)`.

In [ ]:
_RULESHIFT_BINARY_DF = None
_RULESHIFT_RESULT = None
_RULESHIFT_OFFICIAL_RESULT = None


@kbench.task(
    name="ruleshift_benchmark_v1_binary",
    description=(
        "Cognitive flexibility benchmark: infer a hidden rule shift from "
        "sparse contradictory evidence in a sequence of marker records, "
        "then predict four post-shift probe outputs."
    ),
)
def ruleshift_benchmark_v1_binary(llm) -> tuple[int, int]:
    import json

    global _RULESHIFT_BINARY_DF, _RULESHIFT_RESULT, _RULESHIFT_OFFICIAL_RESULT

    rows = []
    for row in leaderboard_df.itertuples(index=False):
        num_correct, total = run_binary_task(
            llm=llm,
            prompt_binary=row.prompt_binary,
            probe_targets=row.probe_targets,
        )
        rows.append(
            {
                "episode_id": row.episode_id,
                "split": row.split,
                "num_correct": int(num_correct),
                "total": int(total),
            }
        )

    binary_df = pd.DataFrame(rows)
    if binary_df.empty:
        raise RuntimeError("evaluation produced no rows")

    numerator = int(binary_df["num_correct"].sum())
    denominator = int(binary_df["total"].sum())
    total_episodes = len(binary_df)
    if denominator == 0:
        raise RuntimeError("evaluation produced a zero denominator")

    split_names = tuple(binary_df["split"].dropna().astype(str).unique())
    result = {
        "score": numerator / denominator,
        "numerator": numerator,
        "denominator": denominator,
        "total_episodes": total_episodes,
        "benchmark_version": MANIFEST_VERSION,
        "split": split_names[0] if len(split_names) == 1 else "frozen_leaderboard",
        "manifest_version": MANIFEST_VERSION,
    }
    official_result = (result["numerator"], result["denominator"])

    _RULESHIFT_BINARY_DF = binary_df
    _RULESHIFT_RESULT = result
    _RULESHIFT_OFFICIAL_RESULT = official_result

    print("=== RuleShift Benchmark Result ===")
    print(json.dumps(result, indent=2))
    print("=== End Result ===")
    return official_result


In [ ]:
score = ruleshift_benchmark_v1_binary(kbench.llm)
result = _RULESHIFT_RESULT
if result is None:
    raise RuntimeError("ruleshift_benchmark_v1_binary did not populate _RULESHIFT_RESULT")


In [ ]:
_result_df = pd.DataFrame(
    [
        ("Final score", f"{result['score'] * 100:.1f}%"),
        ("Correct / Total probes", f"{result['numerator']} / {result['denominator']}"),
        ("Episodes evaluated", result["total_episodes"]),
        ("Submitted split", result["split"]),
        ("Benchmark version", result["benchmark_version"]),
    ],
    columns=["Field", "Value"],
).set_index("Field")

_result_df


In [ ]:
_summary_df = _RULESHIFT_BINARY_DF[_RULESHIFT_BINARY_DF["num_correct"] < _RULESHIFT_BINARY_DF["total"]].copy()
if _summary_df.empty:
    _summary_df = partition_df.copy()
else:
    _summary_df["missed"] = _summary_df["total"] - _summary_df["num_correct"]
    _summary_df = _summary_df[["episode_id", "split", "num_correct", "total", "missed"]]

_summary_df


## Task selection


In [ ]:
%choose ruleshift_benchmark_v1_binary
